<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#라이브러리
import pandas as pd
import re
from urllib.parse import urlparse
from google.colab import drive
import requests

**피싱 URL 데이터**

In [ ]:
#데이터셋 로드
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/malicious_phish.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================
# Label 생성
# ==========================

df["label"] = df["type"].apply(
    lambda x: 0 if x == "benign" else 1
)

# ==========================
# URL Parsing
# ==========================

def parse_url(url):
    try:
        if not str(url).startswith(("http://", "https://")):
            url = "http://" + str(url)

        parsed = urlparse(url)

        return pd.Series({
            "scheme": parsed.scheme,
            "domain": parsed.netloc,
            "path": parsed.path,
            "query": parsed.query
        })
    except ValueError:
        # Handle the error by returning empty values for malformed URLs
        return pd.Series({
            "scheme": "",
            "domain": "",
            "path": "",
            "query": ""
        })

parsed = df["url"].apply(parse_url)

df = pd.concat([df, parsed], axis=1)

# ==========================
# URL Length
# ==========================

df["url_length"] = df["url"].astype(str).apply(len)

# ==========================
# Domain Length
# ==========================

df["domain_length"] = df["domain"].astype(str).apply(len)

# ==========================
# Dot Count
# ==========================

df["dot_count"] = df["url"].astype(str).str.count(r"\.")

# ==========================
# Slash Count
# ==========================

df["slash_count"] = df["url"].astype(str).str.count("/")

# ==========================
# Hyphen Count
# ==========================

df["hyphen_count"] = df["url"].astype(str).str.count("-")

# ==========================
# Digit Count
# ==========================

df["digit_count"] = df["url"].astype(str).apply(
    lambda x: len(re.findall(r"\d", x))
)

# ==========================
# @ 포함 여부
# ==========================

df["contains_at"] = df["url"].astype(str).apply(
    lambda x: 1 if "@" in x else 0
)

# ==========================
# IP 사용 여부
# ==========================

def has_ip(url):
    pattern = r"(?:\d{1,3}\.){3}\d{1,3}"
    return 1 if re.search(pattern, str(url)) else 0

df["has_ip"] = df["url"].apply(has_ip)

# ==========================
# HTTPS 여부
# ==========================

df["https_flag"] = (
    df["scheme"]
    .apply(lambda x: 1 if x == "https" else 0)
)

# ==========================
# 단축 URL
# ==========================

shorteners = [
    "bit.ly",
    "tinyurl",
    "goo.gl",
    "t.co"
]

df["shortener"] = df["url"].apply(
    lambda x: int(
        any(
            s in str(x).lower()
            for s in shorteners
        )
    )
)

# ==========================
# 의심 키워드
# ==========================

keywords = [
    "login",
    "verify",
    "account",
    "secure",
    "update",
    "bank",
    "paypal",
    "apple",
    "facebook"
]

def suspicious_count(url):
    return sum(
        word in str(url).lower()
        for word in keywords
    )

df["suspicious_word_count"] = (
    df["url"].apply(suspicious_count)
)

# 저장
save_path = "/content/drive/MyDrive/REDRED/url_features.csv"
df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print(df.head())

                                                 url        type  label  \
0                                   br-icloud.com.br    phishing      1   
1                mp3raid.com/music/krizz_kaliko.html      benign      0   
2                    bopsecrets.org/rexroth/cr/1.htm      benign      0   
3  http://www.garage-pirenne.be/index.php?option=...  defacement      1   
4  http://adventure-nicaragua.net/index.php?optio...  defacement      1   

  scheme                   domain                      path  \
0   http         br-icloud.com.br                             
1   http              mp3raid.com  /music/krizz_kaliko.html   
2   http           bopsecrets.org         /rexroth/cr/1.htm   
3   http    www.garage-pirenne.be                /index.php   
4   http  adventure-nicaragua.net                /index.php   

                                               query  url_length  \
0                                                             16   
1                                 

**재난 안전 문자 API**

In [ ]:
#API연동
SERVICE_KEY = "API키"

url = "https://www.safetydata.go.kr/V2/api/DSSP-IF-00247"

params = {
    "serviceKey": SERVICE_KEY,
    "pageNo": 1,
    "numOfRows": 1000,
    "returnType": "json",
    "crtDt": "20260708"
}

response = requests.get(url, params=params)

data = response.json()

print(data)

{'header': {'resultMsg': 'NORMAL SERVICE', 'resultCode': '00', 'errorMsg': None}, 'numOfRows': 1000, 'pageNo': 1, 'totalCount': 687, 'body': [{'MSG_CN': '오늘 05:50 미평천 청주시(장성2교)지점 심각단계. 하천범람에 대비 바랍니다. 내 위치, 침수우려지역 확인 vo.la/Vo27SU [금강홍수통제소]', 'RCPTN_RGN_NM': '충청북도 청주시 ', 'CRT_DT': '2026/07/09 05:58:23', 'REG_YMD': '2026/07/09 05:58:48.000000000', 'EMRG_STEP_NM': '긴급재난', 'SN': 260074, 'DST_SE_NM': '홍수', 'MDFCN_YMD': '2026/07/09 06:08:38.000000000'}, {'MSG_CN': '호우주의보 발효 중. 10시 이후 강한 비가 예상됩니다. ▲ 산림·하천·계곡 등 위험지역 출입 금지 ▲ 대피 권고 시 즉시 안전한 곳으로 대피하세요.[여주시]', 'RCPTN_RGN_NM': '경기도 여주시 ', 'CRT_DT': '2026/07/09 09:47:24', 'REG_YMD': '2026/07/09 09:47:48.000000000', 'EMRG_STEP_NM': '안전안내', 'SN': 260197, 'DST_SE_NM': '호우', 'MDFCN_YMD': '2026/07/09 09:57:38.000000000'}, {'MSG_CN': '열대야 주의보 발효 중 ▲취침 전 충분한 수분 섭취 ▲쾌적한 실내 온도 및 습도 조절 ▲시원한 침구사용하기 등 건강수칙을 준수하여 주시기 바랍니다. [달성군]', 'RCPTN_RGN_NM': '대구광역시 달성군 ', 'CRT_DT': '2026/07/10 18:29:14', 'REG_YMD': '2026/07/10 18:29:39.000000000', 'EMRG_STEP_NM': '안전안내', 'SN

In [ ]:
import requests
import pandas as pd
import re

# ==========================================
# 2. DataFrame 생성
# ==========================================

messages = []

for item in data["body"]:
    messages.append({
        "message": item.get("MSG_CN", ""),
        "region": item.get("RCPTN_RGN_NM", ""),
        "disaster_type": item.get("DST_SE_NM", ""),
        "alert_type": item.get("EMRG_STEP_NM", "")
    })

df = pd.DataFrame(messages)

print("수집 건수 :", len(df))

# ==========================================
# 3. 기관명 추출
# ==========================================

df["agency"] = df["message"].str.extract(
    r"\[([^\]]+)\]"
)

# ==========================================
# 4. 기관명 제거
# ==========================================

df["clean_text"] = df["message"].str.replace(
    r"\[[^\]]+\]",
    "",
    regex=True
)

# ==========================================
# 5. URL 추출
# ==========================================

url_pattern = r'https?://[^\s]+|www\.[^\s]+'

df["urls"] = df["message"].apply(
    lambda x: re.findall(url_pattern, str(x))
)

# URL 존재 여부

df["has_url"] = df["urls"].apply(
    lambda x: 1 if len(x) > 0 else 0
)

# URL 개수

df["url_count"] = df["urls"].apply(len)

# ==========================================
# 6. 텍스트 길이
# ==========================================

df["text_length"] = df["clean_text"].str.len()

# ==========================================
# 7. 재난 키워드 Feature
# ==========================================

disaster_keywords = [
    "호우",
    "태풍",
    "폭염",
    "한파",
    "산불",
    "화재",
    "강풍",
    "지진",
    "홍수",
    "붕괴",
    "대설",
    "황사"
]

def disaster_keyword_count(text):
    return sum(
        keyword in str(text)
        for keyword in disaster_keywords
    )

df["disaster_keyword_count"] = (
    df["clean_text"]
    .apply(disaster_keyword_count)
)

# ==========================================
# 8. 스미싱 키워드 Feature
# ==========================================

smishing_keywords = [
    "지원금",
    "보상금",
    "당첨",
    "쿠폰",
    "수령",
    "환급",
    "조회",
    "확인",
    "상품권",
    "이벤트",
    "링크",
    "접속",
    "인증",
    "로그인",
    "계정",
    "결제"
]

def smishing_keyword_count(text):
    return sum(
        keyword in str(text)
        for keyword in smishing_keywords
    )

df["smishing_keyword_count"] = (
    df["clean_text"]
    .apply(smishing_keyword_count)
)

# ==========================================
# 9. 긴급성 키워드 Feature
# ==========================================

emergency_keywords = [
    "긴급",
    "즉시",
    "지금",
    "오늘",
    "신속",
    "필수",
    "주의",
    "경보",
    "발효"
]

def emergency_keyword_count(text):
    return sum(
        keyword in str(text)
        for keyword in emergency_keywords
    )

df["emergency_keyword_count"] = (
    df["clean_text"]
    .apply(emergency_keyword_count)
)

# ==========================================
# 10. 공백 정리
# ==========================================

df["clean_text"] = (
    df["clean_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# ==========================================
# 11. 최종 컬럼 확인
# ==========================================

final_df = df[
    [
        "message",
        "clean_text",
        "agency",
        "region",
        "disaster_type",
        "alert_type",
        "has_url",
        "url_count",
        "text_length",
        "disaster_keyword_count",
        "smishing_keyword_count",
        "emergency_keyword_count"
    ]
]

print(final_df.head())

# ==========================================
# 12. CSV 저장
# ==========================================

save_path = "/content/drive/MyDrive/REDRED/disaster_sms_preprocessed.csv"
final_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료")

수집 건수 : 687
                                             message  \
0  오늘 05:50 미평천 청주시(장성2교)지점 심각단계. 하천범람에 대비 바랍니다. ...   
1  호우주의보 발효 중. 10시 이후 강한 비가 예상됩니다. ▲ 산림·하천·계곡 등 위...   
2  열대야 주의보 발효 중 ▲취침 전 충분한 수분 섭취 ▲쾌적한 실내 온도 및 습도 조...   
3  폭염 대비 안전관리 유의 ▲한낮 야외활동 자제 ▲충분한 수분 섭취 ▲야외활동 시 그...   
4  오늘 11:00 합천중부 폭염경보. 야외 활동 자제, 충분히 물을 마시고 그늘에서 ...   

                                          clean_text   agency      region  \
0  오늘 05:50 미평천 청주시(장성2교)지점 심각단계. 하천범람에 대비 바랍니다. ...  금강홍수통제소   충청북도 청주시    
1  호우주의보 발효 중. 10시 이후 강한 비가 예상됩니다. ▲ 산림·하천·계곡 등 위...      여주시    경기도 여주시    
2  열대야 주의보 발효 중 ▲취침 전 충분한 수분 섭취 ▲쾌적한 실내 온도 및 습도 조...      달성군  대구광역시 달성군    
3  폭염 대비 안전관리 유의 ▲한낮 야외활동 자제 ▲충분한 수분 섭취 ▲야외활동 시 그...      산청군   경상남도 산청군    
4  오늘 11:00 합천중부 폭염경보. 야외 활동 자제, 충분히 물을 마시고 그늘에서 ...      합천군   경상남도 합천군    

  disaster_type alert_type  has_url  url_count  text_length  \
0            홍수       긴급재난        0          0           75   
1            호우       안전안내        0          0           83   

In [ ]:
import pandas as pd
import re

# ==========================================
# 1. 데이터 불러오기
# ==========================================

df = pd.read_csv(
    "/content/drive/MyDrive/source_documents.csv"
)

print("원본 데이터 개수:", len(df))

# ==========================================
# 2. 중복 제거
# ==========================================

df = df.drop_duplicates(subset=["text_hash"])

# ==========================================
# 3. raw_text 없는 데이터 제거
# ==========================================

df = df.dropna(subset=["raw_text"])

print("중복 제거 후:", len(df))

# ==========================================
# 4. 텍스트 정제
# ==========================================

def clean_text(text):

    text = str(text)

    text = re.sub(r'\s+', ' ', text)

    text = text.strip()

    return text

df["clean_text"] = df["raw_text"].apply(
    clean_text
)

# ==========================================
# 5. 재난 유형 추출
# ==========================================

disaster_keywords = [
    "화재",
    "폭염",
    "호우",
    "태풍",
    "수난사고",
    "교통사고",
    "추락사고",
    "산악사고",
    "폭발사고",
    "폭발",
    "산불",
    "가스누출",
    "정전사고",
    "정전",
    "붕괴사고",
    "붕괴",
    "열차사고",
    "물놀이사고",
    "벌쏘임",
    "안전사고"
]

def find_disaster(text):

    found = []

    for keyword in disaster_keywords:

        if keyword in text:
            found.append(keyword)

    if len(found) == 0:
        return "기타"

    return ",".join(list(set(found)))

df["disaster_type"] = (
    df["clean_text"]
    .apply(find_disaster)
)

# ==========================================
# 6. 지역 추출
# ==========================================

regions = [
    "서울",
    "부산",
    "대구",
    "인천",
    "광주",
    "대전",
    "울산",
    "세종",
    "경기",
    "강원",
    "충북",
    "충남",
    "전북",
    "전남",
    "경북",
    "경남",
    "제주"
]

def extract_region(text):

    found = []

    for region in regions:

        if region in text:
            found.append(region)

    if len(found) == 0:
        return "기타"

    return ",".join(list(set(found)))

df["region"] = (
    df["clean_text"]
    .apply(extract_region)
)

# ==========================================
# 7. 사망자 수 추출
# ==========================================

def extract_death(text):

    matches = re.findall(
        r'사망\s*(\d+)명',
        text
    )

    if len(matches) == 0:
        return 0

    return sum(
        map(int, matches)
    )

df["death_count"] = (
    df["clean_text"]
    .apply(extract_death)
)

# ==========================================
# 8. 부상자 수 추출
# ==========================================

def extract_injury(text):

    matches = re.findall(
        r'부상\s*(\d+)명',
        text
    )

    if len(matches) == 0:
        return 0

    return sum(
        map(int, matches)
    )

df["injury_count"] = (
    df["clean_text"]
    .apply(extract_injury)
)

# ==========================================
# 9. URL 개수 추출
# ==========================================

url_pattern = r'https?://[^\s]+'

df["url_count"] = (
    df["clean_text"]
    .apply(
        lambda x: len(
            re.findall(url_pattern, x)
        )
    )
)

# ==========================================
# 10. 문자 길이
# ==========================================

df["text_length"] = (
    df["clean_text"]
    .str.len()
)

# ==========================================
# 11. 위험도 점수(프로젝트용)
# ==========================================

df["risk_score"] = (
    df["death_count"] * 5
    + df["injury_count"] * 2
)

# ==========================================
# 12. 최종 데이터셋
# ==========================================

final_df = df[
    [
        "id",
        "report_date",
        "disaster_type",
        "region",
        "death_count",
        "injury_count",
        "risk_score",
        "text_length",
        "clean_text"
    ]
]

print(final_df.head())

# ==========================================
# 13. 저장
# ==========================================

save_path = (
    "/content/drive/MyDrive/REDRED/"
    "소방청일일상황보고.csv"
)

final_df.to_csv(
    save_path,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 50)
print("전처리 완료")
print("저장 위치:", save_path)
print("=" * 50)

# ==========================================
# 14. 간단한 통계 확인
# ==========================================

print("\n재난 유형 TOP10")
print(
    final_df["disaster_type"]
    .value_counts()
    .head(10)
)

print("\n지역 TOP10")
print(
    final_df["region"]
    .value_counts()
    .head(10)
)

print("\n사망자 총합")
print(
    final_df["death_count"].sum()
)

print("\n부상자 총합")
print(
    final_df["injury_count"].sum()
)

원본 데이터 개수: 3201
중복 제거 후: 2967
     id report_date           disaster_type                   region  \
0  3557  2017-07-26  화재,추락사고,안전사고,물놀이사고,벌쏘임     인천,제주,경남,전남,서울,부산,충남   
1  3558  2017-07-27        호우,화재,교통사고,물놀이사고  인천,경기,경북,강원,전남,서울,부산,대구   
2  3559  2017-07-28    화재,교통사고,물놀이사고,호우,벌쏘임  인천,경기,경북,강원,전남,서울,부산,대구   
3  3560  2017-07-29     폭발,화재,수난사고,정전,물놀이사고     경기,대전,강원,부산,광주,전북,경북   
4  3561  2017-07-30  화재,교통사고,수난사고,물놀이사고,벌쏘임  인천,경기,대전,세종,전남,서울,부산,전북   

   death_count  injury_count  risk_score  text_length  \
0            1            42          89          866   
1            1             8          21          919   
2            1            52         109          952   
3            1             0           5          895   
4            1             5          15          874   

                                          clean_text  
0  소방 안전 활동 상황 총 괄 구 분 화 재 구 조 구 급 건수 사망 부상 피해액 (...  
1  119 소방안전 활동상황 2017.7.28.(금) 06:00 119종합상황실 총 괄...  
2  119 소방안전 활동상황 2017.7.28.(금

In [3]:
import pandas as pd
import re
from urllib.parse import urlparse

# =====================================================
# 파일 로드
# =====================================================

sms = pd.read_csv("/content/drive/MyDrive/REDRED/disaster_sms_preprocessed.csv")
fire = pd.read_csv("/content/drive/MyDrive/REDRED/소방청일일상황보고.csv")

# =====================================================
# URL 탐지 함수
# =====================================================

URL_PATTERN = re.compile(
    r"""
    (
        https?://[^\s\]\)]+
        |
        www\.[^\s\]\)]+
        |
        [a-zA-Z0-9-]+\.
        (?:com|net|org|kr|go\.kr|co\.kr|or\.kr|ac\.kr|la|ly|io|me|tv)
        (?:/[^\s\]\)]*)?
    )
    """,
    re.VERBOSE | re.IGNORECASE
)

SHORTENERS = {
    "bit.ly",
    "vo.la",
    "tinyurl.com",
    "goo.gl",
    "t.co",
    "ow.ly",
    "is.gd",
    "buff.ly",
    "cutt.ly",
    "rb.gy"
}


def extract_urls_from_text(text):

    text = str(text)

    urls = URL_PATTERN.findall(text)

    urls = list(set(urls))

    return urls


def extract_domains(urls):

    domains = []

    for url in urls:

        if not url.startswith(("http://", "https://")):
            temp = "http://" + url
        else:
            temp = url

        try:
            domain = urlparse(temp).netloc.lower()
            domain = domain.replace("www.", "")
            domains.append(domain)
        except:
            pass

    return list(set(domains))


# =====================================================
# SMS URL 재계산
# =====================================================

sms["detected_urls"] = sms["clean_text"].apply(
    extract_urls_from_text
)

sms["url_count_fixed"] = sms["detected_urls"].apply(len)

sms["has_url_fixed"] = (
    sms["url_count_fixed"] > 0
).astype(int)

sms["url_domains"] = (
    sms["detected_urls"]
    .apply(extract_domains)
)

sms["has_shortened_url"] = (
    sms["url_domains"]
    .apply(
        lambda x:
        int(
            any(
                d in SHORTENERS
                for d in x
            )
        )
    )
)

# =====================================================
# 기존 URL 컬럼과 비교
# =====================================================

sms["url_detection_error"] = (
    sms["has_url"] != sms["has_url_fixed"]
).astype(int)

print("=" * 50)
print("URL 누락 건수")
print(
    sms["url_detection_error"].sum()
)

# =====================================================
# 사건 단위 분해
# =====================================================

event_rows = []

event_pattern = r'❍\((.*?)\)'

for _, row in fire.iterrows():

    report_id = row["id"]

    text = str(row["clean_text"])

    matches = list(
        re.finditer(event_pattern, text)
    )

    for i, match in enumerate(matches):

        start = match.start()

        end = (
            matches[i+1].start()
            if i < len(matches)-1
            else len(text)
        )

        block = text[start:end]

        category = match.group(1)

        event_rows.append(
            {
                "report_id": report_id,
                "report_date": row["report_date"],
                "category": category,
                "event_text": block
            }
        )

events = pd.DataFrame(event_rows)

print("=" * 50)
print("사건 수")
print(len(events))

# =====================================================
# 지역 추출
# =====================================================

region_pattern = (
    r'(서울|부산|대구|인천|광주|대전|울산|세종|'
    r'경기|강원|충북|충남|전북|전남|경북|경남|제주)'
)

events["sido"] = (
    events["event_text"]
    .str.extract(region_pattern)
)

# =====================================================
# 사망 / 중상 / 경상
# =====================================================

def extract_num(pattern, text):

    nums = re.findall(pattern, str(text))

    if len(nums) == 0:
        return 0

    return sum(
        int(x)
        for x in nums
        if x.isdigit()
    )

events["death"] = (
    events["event_text"]
    .apply(
        lambda x:
        extract_num(
            r'사망\s*(\d+)명',
            x
        )
    )
)

events["severe"] = (
    events["event_text"]
    .apply(
        lambda x:
        extract_num(
            r'중상\s*(\d+)명',
            x
        )
    )
)

events["minor"] = (
    events["event_text"]
    .apply(
        lambda x:
        extract_num(
            r'경상\s*(\d+)명',
            x
        )
    )
)

# =====================================================
# 사건 사망자 합산
# =====================================================

event_death_sum = (
    events
    .groupby("report_id")["death"]
    .sum()
    .reset_index()
)

event_death_sum.columns = [
    "id",
    "event_death_sum"
]

fire = fire.merge(
    event_death_sum,
    on="id",
    how="left"
)

fire["event_death_sum"] = (
    fire["event_death_sum"]
    .fillna(0)
)

# =====================================================
# 사망자 불일치 검증
# =====================================================

fire["death_mismatch"] = (
    fire["death_count"]
    != fire["event_death_sum"]
).astype(int)

print("=" * 50)
print("사망자 불일치 건수")
print(
    fire["death_mismatch"].sum()
)

print("=" * 50)
print(
    fire.loc[
        fire["death_mismatch"] == 1,
        [
            "id",
            "report_date",
            "death_count",
            "event_death_sum"
        ]
    ].head(20)
)

# =====================================================
# 저장
# =====================================================

sms.to_csv(
    "/content/drive/MyDrive/REDRED/sms_url_fixed.csv",
    index=False,
    encoding="utf-8-sig"
)

events.to_csv(
    "/content/drive/MyDrive/REDRED/fire_events.csv",
    index=False,
    encoding="utf-8-sig"
)

fire.to_csv(
    "/content/drive/MyDrive/REDRED/fire_reports_validated.csv",
    index=False,
    encoding="utf-8-sig"
)

print("완료")

URL 누락 건수
44
사건 수
2135
사망자 불일치 건수
1378
       id report_date  death_count  event_death_sum
10   3500  2017-08-05            2              0.0
11   3498  2017-08-06            2              1.0
25   3468  2017-08-21            4              0.0
61   3380  2017-10-04            3              0.0
70   3356  2017-10-16            3              0.0
257  2820  2018-06-20            2              1.0
276  2737  2018-07-17            1              0.0
304  2629  2018-08-23            2              1.0
351  2430  2018-10-29            4              2.0
385  2290  2018-12-15            2              0.0
387  2284  2018-12-17            2              0.0
390  2272  2018-12-21            2              1.0
391  2269  2018-12-22            1              0.0
393  2260  2018-12-25            2              0.0
397  2242  2018-12-31            1              0.0
398  2239  2019-01-01            1              0.0
399  2236  2019-01-02            2              0.0
400  2230  2019-01-04    